In [1]:
# Lib
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
# Folder path
folder_path = "Data_enhancement"

# Create folder to save clustering results
output_folder_clustering = "Data_cluster_K4"
os.makedirs(output_folder_clustering, exist_ok=True)

# Create folder to save masking results
output_folder_masks = "Data_mask_K4"
os.makedirs(output_folder_masks, exist_ok=True)

# Create folder to save compared result of cluster images
output_compared = "Data_Compared_K4"
os.makedirs(output_compared, exist_ok=True)

# Sorted images
image_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".bmp")])

# Selected cluster for generate masks
selected_cluster = [0]

# Lists to store eval metrics
sse_list = []
dbi_list = []
chi_list = []

# Define dataframe
df = pd.DataFrame(columns=['Image', 'K_Optimum', 'SSE', 'CHI', 'DBI', 'Pore_dist', 'Other_dist'])

In [ ]:
# Initialize tqdm outside the loop
with tqdm(total=len(image_files), desc="Processing images", unit="file") as pbar:
    for i, img_name in enumerate(image_files):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_BAYER_BGGR2RGB)
        assert img is not None, "File could not be read, check with os.path.exists()"

        # --- Flatten image ---
        image_2d = img.reshape((-1, 3))
        image_2d = np.float32(image_2d)

        # Model KMeans
        model = KMeans(random_state=42, n_init=10)

        # Take optimal k from the visualizer
        k_optimal = 4

        # Convert the RGB image to HLS
        img_hls = cv2.cvtColor(img, cv2.COLOR_RGB2HLS)
        lightness = img_hls[:, :, 1]

        # Apply KMeans clustering
        kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
        labels = kmeans.fit_predict(image_2d)

        segmented_img = kmeans.cluster_centers_[labels].reshape(img.shape).astype(np.uint8)
        label_map = labels.reshape(img.shape[0], img.shape[1])

        # --- Sorting clusters by average lightness ---
        average_lightness = []
        for label in range(k_optimal):
            mask = (label_map == label)
            avg_l = np.mean(lightness[mask]) if np.any(mask) else 0
            average_lightness.append((label, avg_l))

        average_lightness.sort(key=lambda x: x[1])
        sorted_labels = {old_label: new_label for new_label, (old_label, _) in enumerate(average_lightness)}

        sorted_label_map = np.copy(label_map)
        for old_label, new_label in sorted_labels.items():
            sorted_label_map[label_map == old_label] = new_label

        sorted_segmented_img = np.zeros_like(segmented_img)
        for label, color in enumerate(kmeans.cluster_centers_.astype(np.uint8)):
            sorted_segmented_img[sorted_label_map == label] = color
       # --- Save clustered result in grayscale (0..255) ---
        sorted_label_map_gray = (sorted_label_map / (k_optimal - 1) * 255).astype(np.uint8)

        output_path_cluster = os.path.join(output_folder_clustering, f"Clustered_{img_name}")
        cv2.imwrite(output_path_cluster, sorted_label_map_gray)

        # --- Masking ---
        selected_cluster_mask = np.isin(sorted_label_map, selected_cluster)

        mask_image = np.zeros_like(img)
        mask_image[selected_cluster_mask] = [255, 255, 255]

        output_path_mask = os.path.join(output_folder_masks, f"Mask_{img_name}")
        cv2.imwrite(output_path_mask, cv2.cvtColor(mask_image, cv2.COLOR_RGB2BGR))

        # Porous distribution
        # Calculate pore area and volume in percentage
        pore_area = np.sum(mask_image == 255)
        other = np.sum(mask_image == 0)
        total_area = pore_area + other
        pore_dist = np.round(100 * pore_area / total_area, 2)
        other_dist = np.round(100 * other / total_area, 2)

        # --- Plotting results ---
        plt.figure(figsize=(20, 6))

        plt.subplot(1, 5, 1)
        plt.imshow(img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(1, 5, 2)
        plt.imshow(label_map, cmap="nipy_spectral")
        plt.title(f"Cluster (k={k_optimal})")
        plt.axis("off")

        plt.subplot(1, 5, 3)
        plt.imshow(sorted_label_map, cmap='gray')
        plt.title("Sorted by Lightness")
        plt.axis("off")

        plt.subplot(1, 5, 4)
        plt.imshow(sorted_label_map, cmap='nipy_spectral')
        plt.title("Clustered Result")
        plt.axis("off")

        plt.subplot(1, 5, 5)
        plt.imshow(mask_image)
        plt.title("Masking")
        plt.axis("off")

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.suptitle(f"Image {i}", fontsize=16, fontweight='bold')

        # Save plot
        output_path_compared = os.path.join(output_compared, f"Plot_{img_name}.png")
        plt.savefig(output_path_compared, dpi=300, bbox_inches='tight')

        plt.close()


        # Evaluation metrics
        sse = kmeans.inertia_
        calinski_harabasz = calinski_harabasz_score(image_2d, labels)
        dbi = davies_bouldin_score(image_2d, labels)
        print(f"Sum of Squared Errors (SSE) Image{i}: {sse}")
        print(f"Davies-Bouldin Index (DBI) Image{i}: {dbi}")
        print(f"Calinski-Harabasz Index Image{i}: {calinski_harabasz}")

        # Store metrics
        sse_list.append(sse)
        dbi_list.append(dbi)
        chi_list.append(calinski_harabasz)

        print("=" * 1000)

        # Insert data to dataframe
        df = pd.concat([
        df,
        pd.DataFrame([{
            "Image": img_name,
            "K_Optimum": k_optimal,
            "SSE": sse,
            "CHI": calinski_harabasz,
            "DBI": dbi,
            "Pore_dist": pore_dist,
            "Other_dist": other_dist
        }])
        ], ignore_index=True)

        # Update progress bar
        pbar.update(1)

Processing images:   0%|          | 1/851 [00:05<1:14:17,  5.24s/file]

Sum of Squared Errors (SSE) Image0: 46507684.0
Davies-Bouldin Index (DBI) Image0: 0.6039376481652419
Calinski-Harabasz Index Image0: 2761872.432161844


Processing images:   0%|          | 2/851 [00:09<1:08:59,  4.88s/file]

Sum of Squared Errors (SSE) Image1: 46352024.0
Davies-Bouldin Index (DBI) Image1: 0.5999424795191957
Calinski-Harabasz Index Image1: 2767843.6033569756


Processing images:   0%|          | 3/851 [00:14<1:08:00,  4.81s/file]

...

Sum of Squared Errors (SSE) Image539: 52158772.0
Davies-Bouldin Index (DBI) Image539: 0.6144900828520823
Calinski-Harabasz Index Image539: 2240051.486693838


Processing images:  64%|██████▎   | 541/851 [42:15<24:35,  4.76s/file]

Sum of Squared Errors (SSE) Image540: 51802744.0
Davies-Bouldin Index (DBI) Image540: 0.6134217002390085
Calinski-Harabasz Index Image540: 2253619.8919667886


Processing images:  64%|██████▎   | 542/851 [42:20<24:49,  4.82s/file]

Sum of Squared Errors (SSE) Image541: 51494156.0
Davies-Bouldin Index (DBI) Image541: 0.612418350568261
Calinski-Harabasz Index Image541: 2260583.8391191945


In [ ]:
# Compute and print average evaluation metrics
avg_sse = np.mean(sse_list)
avg_dbi = np.mean(dbi_list)
avg_chi = np.mean(chi_list)

print(f"\nSegmentation done for all images!")
print(f"Average SSE: {avg_sse:.2f}")
print(f"Average DBI: {avg_dbi:.2f}")
print(f"Average CHI: {avg_chi:.2f}")

df.to_excel("outputk4.xlsx", index=False)

In [ ]:
# Path folder tempat 851 gambar masking disimpan
folder_path = 'Data_mask_K4'  

total_foreground = 0
total_background = 0

# Loop semua gambar di folder
for filename in os.listdir(folder_path):
    if filename.endswith(('.png', '.jpg', '.bmp')):
        img_path = os.path.join(folder_path, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Hitung jumlah pixel putih (foreground) dan hitam (background)
        foreground = np.sum(img == 255)
        background = np.sum(img == 0)

        total_foreground += foreground
        total_background += background

# Total semua pixel
total_pixels = total_foreground + total_background

# Hitung proporsi
foreground_percent = (total_foreground / total_pixels) * 100
background_percent = (total_background / total_pixels) * 100

print(f"Foreground (white) Porosity: {foreground_percent:.2f}%")
print(f"Background (black) Non Porosity: {background_percent:.2f}%")